# Practice: time series and forecasting
*AI from scratch, elective* · lesson: https://ihor.sh/ai/time-series.html

Five years of daily NYC subway ridership: find its strongest repeating period, score the seasonal naive forecast,
build lag and calendar features for a chosen horizon, beat the baseline with boosted trees, backtest walking
forward, and check whether a prediction interval really covers what it claims. Runs on a CPU in seconds.
Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)` are
there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['corr1 = s["2023":].autocorr(1)', 'corr7 = s["2023":].autocorr(7)'], 2: ['pred = s.shift(7).loc[test.index]', 'naive_mae = np.mean(np.abs(pred - test))'], 3: ['X["lag_h"] = s.shift(horizon); X["lag_h7"] = s.shift(horizon + 7); X["lag_h14"] = s.shift(horizon + 14)', 'X["mean_week"] = s.shift(horizon).rolling(7).mean(); then return X'], 4: ['model = HistGradientBoostingRegressor(random_state=0).fit(F[train_rows], s[train_rows])', 'model_mae = np.mean(np.abs(model.predict(F[test_rows]) - s[test_rows]))'], 5: ['folds = [] then for start, end in MONTHS: tr = ok & (F.index >= "2022-01-01") & (F.index < start); te = ok & (F.index >= start) & (F.index < end)', 'fit a new HistGradientBoostingRegressor on tr, then folds.append((start, model MAE on te, seasonal naive MAE on te))'], 6: ['inside = (s[test_rows] >= pred + lo) & (s[test_rows] <= pred + hi)', 'coverage = inside.mean()'], 7: ['maes = [] then for h in [1, 7, 28, 56]: Fh = features(s, h); okh = Fh.notna().all(axis=1); masks as in section 4 with okh', 'fit on the train mask, append the MAE on the test mask']}
_ANSWERS = {1: 'Y29ycjEsIGNvcnI3ID0gc1siMjAyMyI6XS5hdXRvY29ycigxKSwgc1siMjAyMyI6XS5hdXRvY29ycig3KQ==', 2: 'bmFpdmVfbWFlID0gbnAubWVhbihucC5hYnMocy5zaGlmdCg3KS5sb2NbdGVzdC5pbmRleF0gLSB0ZXN0KSk=', 3: 'WFsibGFnX2giXSA9IHMuc2hpZnQoaG9yaXpvbikKWFsibGFnX2g3Il0gPSBzLnNoaWZ0KGhvcml6b24gKyA3KQpYWyJsYWdfaDE0Il0gPSBzLnNoaWZ0KGhvcml6b24gKyAxNCkKWFsibWVhbl93ZWVrIl0gPSBzLnNoaWZ0KGhvcml6b24pLnJvbGxpbmcoNykubWVhbigp', 4: 'bW9kZWwgPSBIaXN0R3JhZGllbnRCb29zdGluZ1JlZ3Jlc3NvcihyYW5kb21fc3RhdGU9MCkuZml0KEZbdHJhaW5fcm93c10sIHNbdHJhaW5fcm93c10pCm1vZGVsX21hZSA9IG5wLm1lYW4obnAuYWJzKG1vZGVsLnByZWRpY3QoRlt0ZXN0X3Jvd3NdKSAtIHNbdGVzdF9yb3dzXSkp', 5: 'Zm9sZHMgPSBbXQpmb3Igc3RhcnQsIGVuZCBpbiBNT05USFM6CiAgICB0ciA9IG9rICYgKEYuaW5kZXggPj0gIjIwMjItMDEtMDEiKSAmIChGLmluZGV4IDwgc3RhcnQpCiAgICB0ZSA9IG9rICYgKEYuaW5kZXggPj0gc3RhcnQpICYgKEYuaW5kZXggPCBlbmQpCiAgICBtID0gSGlzdEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3IocmFuZG9tX3N0YXRlPTApLmZpdChGW3RyXSwgc1t0cl0pCiAgICBmb2xkcy5hcHBlbmQoKHN0YXJ0LCBucC5tZWFuKG5wLmFicyhtLnByZWRpY3QoRlt0ZV0pIC0gc1t0ZV0pKSwgbnAubWVhbihucC5hYnMocy5zaGlmdCg3KVt0ZV0gLSBzW3RlXSkpKSk=', 6: 'aW5zaWRlID0gKHNbdGVzdF9yb3dzXSA+PSBwcmVkICsgbG8pICYgKHNbdGVzdF9yb3dzXSA8PSBwcmVkICsgaGkpCmNvdmVyYWdlID0gaW5zaWRlLm1lYW4oKQ==', 7: 'bWFlcyA9IFtdCmZvciBoIGluIFsxLCA3LCAyOCwgNTZdOgogICAgRmggPSBmZWF0dXJlcyhzLCBoKTsgb2toID0gRmgubm90bmEoKS5hbGwoYXhpcz0xKQogICAgdHJoID0gb2toICYgKEZoLmluZGV4ID49ICIyMDIyLTAxLTAxIikgJiAoRmguaW5kZXggPCAiMjAyNC0xMC0wMSIpOyB0ZWggPSBva2ggJiAoRmguaW5kZXggPj0gIjIwMjQtMTAtMDEiKQogICAgbWFlcy5hcHBlbmQobnAubWVhbihucC5hYnMoSGlzdEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3IocmFuZG9tX3N0YXRlPTApLmZpdChGaFt0cmhdLCBzW3RyaF0pLnByZWRpY3QoRmhbdGVoXSkgLSBzW3RlaF0pKSk='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import HistGradientBoostingRegressor

s = pd.read_csv("https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/ridership.csv", parse_dates=["date"]).set_index("date")["subway"]
print(len(s), "days,", s.index.min().date(), "to", s.index.max().date())
s["2024-09":"2024-11"].plot(figsize=(9, 3), title="Subway riders per day, fall 2024"); plt.show()

## 1. The strongest repeating period
`s.autocorr(k)` is the correlation between the series and itself shifted by `k` days. On 2023 onward (`s["2023":]`),
set `corr1` (lag 1) and `corr7` (lag 7).

In [ ]:
corr1 = corr7 = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print("lag 1:", corr1, "· lag 7:", corr7)

In [ ]:
check(1, lambda: corr7 > 0.8 and corr1 < 0.6, "0.83 a week apart vs 0.49 a day apart: the week is the rhythm to copy")

## 2. The seasonal naive forecast
Forecast each day as the same weekday a week before: `s.shift(7)`. Set `naive_mae`: the mean absolute error over the
test window from October 1, 2024 (`test = s["2024-10-01":]`; select the forecast's same dates with `.loc[test.index]`).

In [ ]:
test = s["2024-10-01":]
naive_mae = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print("seasonal naive MAE:", naive_mae)

In [ ]:
check(2, lambda: abs(naive_mae - 381_582) < 2_000, "about 382,000 riders a day: 11% of an average day, and the number to beat")

## 3. Features for a forecast `horizon` days ahead
Write `features(s, horizon)`: a DataFrame with the same index and columns `lag_h`, `lag_h7`, `lag_h14` (riders
`horizon`, `horizon + 7`, `horizon + 14` days before: `s.shift(...)`) and `mean_week` (the 7-day average ending
`horizon` days before: `s.shift(horizon).rolling(7).mean()`). The calendar columns (weekday, month, day of year,
holiday) are given.

In [ ]:
HOLIDAYS = USFederalHolidayCalendar().holidays(s.index.min(), s.index.max())
def features(s, horizon=7):
    X = pd.DataFrame(index=s.index)
    X["weekday"], X["month"], X["day_of_year"] = s.index.dayofweek, s.index.month, s.index.dayofyear
    X["holiday"] = s.index.isin(HOLIDAYS).astype(int)
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...
    return X

In [ ]:
F = features(s, 7)
check(3, lambda: F.loc["2024-10-15", "lag_h"] == s["2024-10-08"] and np.isclose(F.loc["2024-10-15", "mean_week"], s["2024-10-02":"2024-10-08"].mean()),
      "every feature for October 15 comes from October 8 or earlier: nothing from the week being forecast")

## 4. Beat the baseline
Train `HistGradientBoostingRegressor(random_state=0)` on 2022-01-01 to 2024-09-30 (rows where every feature exists:
`F.notna().all(axis=1)`), predict the test window, and set `model_mae`.

In [ ]:
ok = F.notna().all(axis=1)
train_rows = ok & (F.index >= "2022-01-01") & (F.index < "2024-10-01")
test_rows = ok & (F.index >= "2024-10-01")
model_mae = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("boosted trees MAE:", model_mae, "· seasonal naive:", naive_mae)

In [ ]:
check(4, lambda: model_mae < 0.9 * naive_mae, "clearly better than last week, by a margin you can state")

## 5. Walk forward
One test window is one roll of the dice. For each `(start, end)` in `MONTHS` (given), train on everything from 2022
up to `start` and test from `start` to `end`. Set `folds`: a list of `(start, model MAE, naive MAE)`.

In [ ]:
MONTHS = [("2024-06-01", "2024-07-01"), ("2024-07-01", "2024-08-01"), ("2024-08-01", "2024-09-01"), ("2024-09-01", "2024-10-01"),
          ("2024-10-01", "2024-11-01"), ("2024-11-01", "2024-12-01"), ("2024-12-01", "2025-01-01")]
folds = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
pd.DataFrame(folds, columns=["month", "model MAE", "naive MAE"]).round(0) if folds else None

In [ ]:
check(5, lambda: len(folds) == 7 and sum(m < n for _, m, n in folds) >= 6,
      "it wins every month, by a lot in June, July and December and barely in August and October: the backtest shows where the margin is real")

## 6. Does the 90% interval cover 90%?
A simple interval: the model's prediction plus the 5th and 95th percentiles of its errors on a validation window.
Given: errors from a model trained before July 2024, measured on July–September 2024. Set `coverage`: the share of
test days whose real value falls inside `[pred + lo, pred + hi]`.

In [ ]:
fit_rows = ok & (F.index >= "2022-01-01") & (F.index < "2024-07-01")
val_rows = ok & (F.index >= "2024-07-01") & (F.index < "2024-10-01")
m_val = HistGradientBoostingRegressor(random_state=0).fit(F[fit_rows], s[fit_rows])
lo, hi = np.quantile(s[val_rows] - m_val.predict(F[val_rows]), [0.05, 0.95])
pred = m_val.predict(F[test_rows])
coverage = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print("claimed 90%, actual:", coverage)

In [ ]:
check(6, lambda: coverage < 0.8, "well under 90%: summer's errors were calmer than the holiday season's. Build intervals from comparable seasons, and check coverage")

## 7. How far ahead?
For horizons 1, 7, 28 and 56 days, rebuild the features, train and test as in section 4, and set `maes`: the four MAEs
in that order.

In [ ]:
maes = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print("MAE 1, 7, 28, 56 days ahead:", np.round(maes) if maes else None)

In [ ]:
check(7, lambda: maes[0] < maes[1] and maes[3] > maes[0],
      "worse the further ahead overall (227k a day ahead, 312k at 56 days), though not at every step: 28 days happened to beat 7 in this one window")

## Next
Back to the path: https://ihor.sh/ai/ · Hyndman & Athanasopoulos, "Forecasting: Principles and Practice": https://otexts.com/fpp3/